# V3 : Multi-Satellite Constellation

In [66]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [67]:
import sys

target_path = "/content/drive/MyDrive/GNSS_Spoofing_Research/src"
incorrect_path_variant = "/content/drive/GNSS_Spoofing_Research/src" # Old incorrect path

# Remove any existing instances of the target path or its incorrect variant
sys.path = [p for p in sys.path if p != target_path and p != incorrect_path_variant]

# Insert the correct path at the beginning
sys.path.insert(0, target_path)

print(f"sys.path has been updated. '{target_path}' is now at the beginning.")

sys.path has been updated. '/content/drive/MyDrive/GNSS_Spoofing_Research/src' is now at the beginning.


In [68]:
import os

# Construct the full path to the directory
directory_path = '/content/drive/MyDrive/GNSS_Spoofing_Research/src'

# Check if the directory exists
if os.path.exists(directory_path):
    print(f"Contents of {directory_path}:")
    # List all files and directories in the specified path
    for item in os.listdir(directory_path):
        print(item)
else:
    print(f"The directory {directory_path} does not exist. Please ensure your Google Drive is mounted and the path is correct.")

Contents of /content/drive/MyDrive/GNSS_Spoofing_Research/src:
orbit.py
propagation.py
doppler.py
ranging.py
coordinates.py
__pycache__


In [69]:
import sys
print("Current sys.path before import:")
for p in sys.path:
    print(f"  {p}")

# Attempt the import again
from coordinates import propagate_orbit
print("Successfully imported propagate_orbit from coordinates.py")

Current sys.path before import:
  /content/drive/MyDrive/GNSS_Spoofing_Research/src
  /content
  /env/python
  /usr/lib/python313.zip
  /usr/lib/python3.13
  /usr/lib/python3.13/lib-dynload
  
  /usr/local/lib/python3.13/dist-packages
  /usr/lib/python3/dist-packages
  /usr/local/lib/python3.13/dist-packages/IPython/extensions
  /root/.ipython
Successfully imported propagate_orbit from coordinates.py


In [70]:
# Read and print the content of coordinates.py
file_path = '/content/drive/MyDrive/GNSS_Spoofing_Research/src/coordinates.py'

try:
    with open(file_path, 'r') as f:
        content = f.read()
    print(f"--- Content of {file_path} ---")
    print(content)
    print(f"----------------------------------")
except FileNotFoundError:
    print(f"Error: The file {file_path} was not found. Please re-check the path.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

--- Content of /content/drive/MyDrive/GNSS_Spoofing_Research/src/coordinates.py ---
import numpy as np
import matplotlib.pyplot as plt

#Defining earth
# Earth parameters

R_EARTH = 6_378_137.0       # Earth radius [m](We're using the WGS-84 equatorial radius for now.)
MU_EARTH = 3.986004418e14   # Earth's gravitational parameter [m^3/s^2]

# Satellite altitudes

LEO_ALTITUDE = 600e3       # 600 km
MEO_ALTITUDE = 20_200e3    # 20,200 km

# V2 : Full orbital orientaion

           
def solve_keplers_equation(M, e, tolerance=1e-12, max_iterations=100):
    """
    Solve M = E - e*sin(E) for eccentric anomaly E.
    """

    # Initial guess
    E = M if e < 0.8 else np.pi

    for _ in range(max_iterations):

        f = E - e * np.sin(E) - M
        f_prime = 1 - e * np.cos(E)

        delta_E = f / f_prime
        E -= delta_E

        if abs(delta_E) < tolerance:
            break

    return E

def eccentric_to_true_anomaly(E, e):

    numerator = np.sqrt(1 + e) * np.sin(E / 2)
    de

In [71]:
import numpy as np
import matplotlib.pyplot as plt

In [72]:
#Defining earth
# Earth parameters

R_EARTH = 6_378_137.0       # Earth radius [m](We're using the WGS-84 equatorial radius for now.)
MU_EARTH = 3.986004418e14   # Earth's gravitational parameter [m^3/s^2]

In [73]:
# Satellite altitudes

LEO_ALTITUDE = 600e3       # 600 km
MEO_ALTITUDE = 20_200e3    # 20,200 km

In [74]:
LEO_PHASE = 0.0
MEO_PHASE = np.deg2rad(90)

In [75]:
MEO_a = R_EARTH + 20_200e3
MEO_e = 0.01
MEO_i = np.deg2rad(55)
MEO_raan = np.deg2rad(120)
MEO_arg_periapsis = np.deg2rad(45)
MEO_true_anomaly_0 = np.deg2rad(180)

In [76]:
LEO_a = R_EARTH + 600e3
LEO_e = 0.01

LEO_i = np.deg2rad(53)

LEO_raan = np.deg2rad(40)

LEO_arg_periapsis = np.deg2rad(30)

LEO_true_anomaly_0 = np.deg2rad(0)

## GPS-like constellation

Let's start with:

MEO


* Number of satellites = 24
* Number of orbital planes = 6
* Satellites per plane = 4
* Altitude ≈ 20,200 km
* Inclination = 55°



**This follows the basic GPS constellation structure of six planes and four baseline slots per plane.**

For our first simplified model, we'll distribute:

$$ \Omega = 0°,60°,120°,180°,240°,300° $$

between the six planes.

Within each plane:

$$ \nu_0 = 0°,90°,180°,270° $$

So visually:

* Plane 1:  GPS-01  GPS-02  GPS-03  GPS-04
* Plane 2:  GPS-05  GPS-06  GPS-07  GPS-08
* Plane 3:  GPS-09  GPS-10  GPS-11  GPS-12
* Plane 4:  GPS-13  GPS-14  GPS-15  GPS-16
* Plane 5:  GPS-17  GPS-18  GPS-19  GPS-20
* Plane 6:  GPS-21  GPS-22  GPS-23  GPS-24

This is deliberately a clean research geometry, not an exact reproduction of current GPS slot assignments.

## LEO constellation

We eventually want to investigate:

How many LEO satellites are needed before their Doppler/range measurements provide useful independent positioning geometry?

So let's make the LEO constellation configurable.

For V3, start with:

* LEO satellites = 6
* LEO planes = 2
* 3 satellites per plane
* altitude = 600 km
* inclination = 53°

We'll eventually experiment with:

1. 6 LEO
2. 12 LEO
3. 24 LEO
4. 48 LEO
...

That becomes useful research data later.

In [77]:
gps_satellite = {
    "name": "GPS-01",
    "a": MEO_a,
    "e": MEO_e,
    "inclination": MEO_i,
    "raan": MEO_raan,
    "arg_periapsis": MEO_arg_periapsis,
    "true_anomaly_0": MEO_true_anomaly_0
}

In [78]:
leo_satellite = {
    "name": "LEO-01",
    "a": LEO_a,
    "e": LEO_e,
    "inclination": LEO_i,
    "raan": LEO_raan,
    "arg_periapsis": LEO_arg_periapsis,
    "true_anomaly_0": LEO_true_anomaly_0
}

In [79]:
def create_leo_constellation(
    num_planes=2,
    satellites_per_plane=3
):

    constellation = []

    for plane in range(num_planes):

        raan = np.deg2rad(
            plane * 360 / num_planes
        )

        for sat in range(satellites_per_plane):

            true_anomaly = np.deg2rad(
                sat * 360 / satellites_per_plane
            )

            satellite = {
                "name": f"LEO-{len(constellation)+1:02d}",

                "plane": plane,

                "a": R_EARTH + 600e3,
                "e": 0.01,
                "inclination": np.deg2rad(53),

                "raan": raan,
                "arg_periapsis": 0.0,
                "true_anomaly_0": true_anomaly
            }

            constellation.append(satellite)

    return constellation

In [80]:
def create_gps_constellation(
    num_planes=6,
    satellites_per_plane=4
):

    constellation = []

    for plane in range(num_planes):

        raan = np.deg2rad(
            plane * 360 / num_planes
        )

        for sat in range(satellites_per_plane):

            true_anomaly = np.deg2rad(
                sat * 360 / satellites_per_plane
            )

            satellite = {
                "name": f"GPS-{len(constellation)+1:02d}",

                "plane": plane,

                "a": R_EARTH + 20_200e3,
                "e": 0.01,
                "inclination": np.deg2rad(55),

                "raan": raan,
                "arg_periapsis": 0.0,
                "true_anomaly_0": true_anomaly
            }

            constellation.append(satellite)

    return constellation

In [81]:
gps_constellation = create_gps_constellation()
print(len(gps_constellation))

24


In [82]:
leo_constellation = create_leo_constellation()
print(len(leo_constellation))

6


In [83]:
from coordinates import propagate_orbit

In [84]:
def propagate_constellation(constellation, times):

    constellation_positions = {}
    constellation_velocities = {}

    for satellite in constellation:

        positions, velocities = propagate_orbit(
            satellite["a"],
            satellite["e"],
            satellite["inclination"],
            satellite["raan"],
            satellite["arg_periapsis"],
            satellite["true_anomaly_0"],
            times
        )

        name = satellite["name"]

        constellation_positions[name] = positions
        constellation_velocities[name] = velocities

    return constellation_positions, constellation_velocities

In [85]:
# Simulation time: 24 hours
simulation_time = 24 * 3600

# Number of simulation samples
num_steps = 3000

times = np.linspace(
    0,
    simulation_time,
    num_steps
)

In [86]:
gps_positions, gps_velocities = propagate_constellation(
    gps_constellation,
    times
)

In [87]:
leo_positions, leo_velocities = propagate_constellation(
    leo_constellation,
    times
)

In [88]:
leo_positions["LEO-01"]

array([[ 6908355.63      ,        0.        ,        0.        ],
       [ 6904889.89337307,   132333.53085043,   175612.52684181],
       [ 6894496.26503966,   264534.28800229,   351048.85703448],
       ...,
       [ 5088055.87139841, -2828460.30255989, -3753493.597671  ],
       [ 5233248.40212359, -2729495.79631228, -3622163.26213089],
       [ 5373228.16195978, -2627812.4926057 , -3487224.96050181]])

In [89]:
gps_positions["GPS-01"]

array([[26312355.63      ,        0.        ,        0.        ],
       [26312116.70466924,    64636.37220668,    92310.30613001],
       [26311399.93314617,   129271.57057235,   184618.93584133],
       ...,
       [26309556.9899696 ,   221212.13085472,   315923.66374735],
       [26307682.6994342 ,   285839.61952742,   408221.28287601],
       [26305330.64564636,   350461.91718216,   500511.48846272]])

In [90]:
import plotly.graph_objects as go

fig = go.Figure()

for satellite_name, positions in gps_positions.items():

    fig.add_trace(
        go.Scatter3d(
            x=positions[:, 0] / 1e3,
            y=positions[:, 1] / 1e3,
            z=positions[:, 2] / 1e3,
            mode="lines",
            name=satellite_name,
            line=dict(width=3)
        )
    )

fig.update_layout(
    width=1000,
    height=1000,
    scene=dict(
        xaxis_title="X [km]",
        yaxis_title="Y [km]",
        zaxis_title="Z [km]"
    ),
    title="V3 — GPS-like MEO Constellation"
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [91]:
import plotly.graph_objects as go

fig = go.Figure()

for satellite_name, positions in leo_positions.items():

    fig.add_trace(
        go.Scatter3d(
            x=positions[:, 0] / 1e3,
            y=positions[:, 1] / 1e3,
            z=positions[:, 2] / 1e3,
            mode="lines",
            name=satellite_name,
            line=dict(width=3)
        )
    )

fig.update_layout(
    width=1000,
    height=1000,
    scene=dict(
        xaxis_title="X [km]",
        yaxis_title="Y [km]",
        zaxis_title="Z [km]"
    ),
    title="V3 — LEO Constellation"
)

fig.show()

In [92]:
import numpy as np
import plotly.graph_objects as go


fig = go.Figure()


# ============================================================
# EARTH
# ============================================================

u = np.linspace(0, 2 * np.pi, 50)
v = np.linspace(0, np.pi, 25)

x = R_EARTH / 1e3 * np.outer(
    np.cos(u),
    np.sin(v)
)

y = R_EARTH / 1e3 * np.outer(
    np.sin(u),
    np.sin(v)
)

z = R_EARTH / 1e3 * np.outer(
    np.ones(np.size(u)),
    np.cos(v)
)

fig.add_trace(
    go.Surface(
        x=x,
        y=y,
        z=z,
        opacity=0.25,
        showscale=False,
        name="Earth"
    )
)


# ============================================================
# ORBIT TRACKS
# ============================================================

# ------------------------------------------------------------
# GPS / MEO tracks
# ------------------------------------------------------------

for satellite_name, positions in gps_positions.items():

    fig.add_trace(
        go.Scatter3d(
            x=positions[:, 0] / 1e3,
            y=positions[:, 1] / 1e3,
            z=positions[:, 2] / 1e3,
            mode="lines",
            line=dict(width=2),
            name=satellite_name,
            legendgroup="GPS",
            showlegend=False
        )
    )


# ------------------------------------------------------------
# LEO tracks
# ------------------------------------------------------------

for satellite_name, positions in leo_positions.items():

    fig.add_trace(
        go.Scatter3d(
            x=positions[:, 0] / 1e3,
            y=positions[:, 1] / 1e3,
            z=positions[:, 2] / 1e3,
            mode="lines",
            line=dict(width=2),
            name=satellite_name,
            legendgroup="LEO",
            showlegend=False
        )
    )


# ============================================================
# CURRENT SATELLITE POSITIONS
# ============================================================

time_index = 0


# ------------------------------------------------------------
# GPS / MEO SATELLITES
# ------------------------------------------------------------

gps_x = []
gps_y = []
gps_z = []
gps_labels = []

for satellite in gps_constellation:

    name = satellite["name"]

    position = gps_positions[name][time_index]

    gps_x.append(position[0] / 1e3)
    gps_y.append(position[1] / 1e3)
    gps_z.append(position[2] / 1e3)
    gps_labels.append(name)


fig.add_trace(
    go.Scatter3d(
        x=gps_x,
        y=gps_y,
        z=gps_z,
        mode="markers+text",
        text=gps_labels,
        textposition="top center",
        marker=dict(
            symbol="square",
            size=6
        ),
        name="GPS / MEO Satellites"
    )
)


# ------------------------------------------------------------
# LEO SATELLITES
# ------------------------------------------------------------

leo_x = []
leo_y = []
leo_z = []
leo_labels = []

for satellite in leo_constellation:

    name = satellite["name"]

    position = leo_positions[name][time_index]

    leo_x.append(position[0] / 1e3)
    leo_y.append(position[1] / 1e3)
    leo_z.append(position[2] / 1e3)
    leo_labels.append(name)


fig.add_trace(
    go.Scatter3d(
        x=leo_x,
        y=leo_y,
        z=leo_z,
        mode="markers+text",
        text=leo_labels,
        textposition="top center",
        marker=dict(
            symbol="square",
            size=7
        ),
        name="LEO Satellites"
    )
)


# ============================================================
# CONNECT SATELLITES IN SAME ORBITAL PLANE
# ============================================================

# ------------------------------------------------------------
# LEO CONNECTIONS
# ------------------------------------------------------------

for plane in range(2):

    plane_satellites = [
        sat for sat in leo_constellation
        if sat["plane"] == plane
    ]

    positions = []

    for sat in plane_satellites:

        name = sat["name"]

        position = leo_positions[name][time_index]

        positions.append(position / 1e3)

    positions = np.array(positions)

    fig.add_trace(
        go.Scatter3d(
            x=positions[:, 0],
            y=positions[:, 1],
            z=positions[:, 2],
            mode="lines",
            line=dict(width=5),
            name=f"LEO Plane {plane}",
            legendgroup=f"LEO Plane {plane}"
        )
    )


# ------------------------------------------------------------
# GPS / MEO CONNECTIONS
# ------------------------------------------------------------

for plane in range(6):

    plane_satellites = [
        sat for sat in gps_constellation
        if sat["plane"] == plane
    ]

    positions = []

    for sat in plane_satellites:

        name = sat["name"]

        position = gps_positions[name][time_index]

        positions.append(position / 1e3)

    positions = np.array(positions)

    fig.add_trace(
        go.Scatter3d(
            x=positions[:, 0],
            y=positions[:, 1],
            z=positions[:, 2],
            mode="lines",
            line=dict(width=5),
            name=f"GPS Plane {plane}",
            legendgroup=f"GPS Plane {plane}"
        )
    )


# ============================================================
# LAYOUT
# ============================================================

fig.update_layout(

    title="V3 — LEO + GPS-like MEO Constellation",

    scene=dict(

        xaxis=dict(
            title="X [km]"
        ),

        yaxis=dict(
            title="Y [km]"
        ),

        zaxis=dict(
            title="Z [km]"
        ),

        aspectmode="data"
    ),

    width=1000,
    height=900,

    legend=dict(
        x=0.02,
        y=0.98
    )
)


# ============================================================
# SHOW
# ============================================================

fig.show()

Output hidden; open in https://colab.research.google.com to view.